In [1]:
# RAG-based Worldwide Scholarship & Mobility Advisor

import numpy as np
from typing import List, Dict

from sentence_transformers import SentenceTransformer
import faiss

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
import textwrap

print("Libraries imported.")


Libraries imported.


In [ ]:
knowledge_base = [
    # --- Türkiye + Erasmus ---
    {
        "id": 1,
        "title": "Erasmus – Program Overview",
        "content": (
            "Erasmus is a European student mobility program enabling students to study or intern abroad "
            "for one or two semesters. It promotes cultural exchange, academic development, and cooperation."
        )
    },
    {
        "id": 2,
        "title": "Erasmus – Minimum Academic Requirements",
        "content": (
            "Most universities require a minimum GPA between 2.20 and 2.50 out of 4.00 for Erasmus. "
            "Some faculties impose higher internal thresholds. Students must be enrolled at an eligible institution."
        )
    },
    {
        "id": 3,
        "title": "Erasmus – Grant Categories and Country Groups",
        "content": (
            "Erasmus mobility grants are grouped by the cost of living in the destination country. "
            "Group 1 countries offer higher monthly grants than Group 2 and Group 3. The grant is not meant "
            "to fully cover all living expenses."
        )
    },
    {
        "id": 4,
        "title": "Erasmus – Language Requirements",
        "content": (
            "Erasmus often requires proof of English or local language proficiency, such as B2 level via "
            "IELTS, TOEFL, or institutional exams. Some host universities may ask for additional documents."
        )
    },
    {
        "id": 5,
        "title": "Erasmus – Required Application Documents",
        "content": (
            "Typical application documents include transcript, application form, language certificate, "
            "CV, motivation letter, and preference list of universities. Some institutions also require interviews."
        )
    },

    # --- Mevlana (Türkiye) ---
    {
        "id": 6,
        "title": "Mevlana Exchange Program – Overview",
        "content": (
            "Mevlana is a Turkish national exchange program supporting mobility between Turkish universities "
            "and higher education institutions outside Europe, including Asia, Africa, and the Americas."
        )
    },
    {
        "id": 7,
        "title": "Mevlana – Academic Requirements",
        "content": (
            "Mevlana requires GPA thresholds specified in each yearly call. Different levels (undergraduate, "
            "graduate, doctoral) may have different minimum GPA requirements and language expectations."
        )
    },
    {
        "id": 8,
        "title": "Mevlana – Financial Support",
        "content": (
            "Mevlana provides monthly grants for outgoing and sometimes incoming students. Grant amounts depend "
            "on destination region and annual budgets and are intended to partially support living costs."
        )
    },
    {
        "id": 9,
        "title": "Mevlana – Selection Criteria",
        "content": (
            "Selection is based on academic performance, language skills, motivation letters, interviews, "
            "and sometimes priority fields or partner universities defined by the call."
        )
    },

    # --- Farabi (Türkiye) ---
    {
        "id": 10,
        "title": "Farabi Exchange Program – Overview",
        "content": (
            "Farabi supports mobility between Turkish universities for one or two semesters. Students remain "
            "registered at their home institution, and courses are recognized through learning agreements."
        )
    },
    {
        "id": 11,
        "title": "Farabi – Academic Requirements",
        "content": (
            "Farabi calls specify minimum GPA requirements for associate, undergraduate and graduate students. "
            "Applicants submit transcripts, application forms, and sometimes recommendation letters."
        )
    },
    {
        "id": 12,
        "title": "Farabi – Financial Conditions",
        "content": (
            "Farabi provides monthly scholarships funded nationally. Grant amounts vary by year and are linked "
            "to performance, such as successfully passing a certain proportion of courses."
        )
    },

    # --- TÜBİTAK (Türkiye) ---
    {
        "id": 13,
        "title": "TÜBİTAK 2209-A Undergraduate Research Support",
        "content": (
            "TÜBİTAK 2209-A funds undergraduate research projects supervised by academic advisors. "
            "It covers materials, small equipment, and basic research costs."
        )
    },
    {
        "id": 14,
        "title": "TÜBİTAK – Required Documents",
        "content": (
            "Applications typically include a project proposal, budget, advisor approval, student info forms, "
            "and institutional confirmation. Calls define exact templates and deadlines."
        )
    },
    {
        "id": 15,
        "title": "TÜBİTAK – General Eligibility",
        "content": (
            "Eligible applicants are undergraduate students in Türkiye with acceptable academic standing. "
            "Selection considers originality, feasibility, methodology, and advisor evaluation."
        )
    },

    # --- Erasmus Mundus + EU mobility ---
    {
        "id": 16,
        "title": "Erasmus Mundus Joint Master’s (EMJM) – Overview",
        "content": (
            "EMJM programs are joint master's degrees offered by consortia of European universities. "
            "Students study in at least two countries and receive a joint or multiple degree."
        )
    },
    {
        "id": 17,
        "title": "EMJM – Financial Package",
        "content": (
            "Erasmus Mundus scholarships usually cover full tuition fees, a monthly stipend, travel costs, "
            "and installation support. The exact amounts and rules vary by consortium."
        )
    },
    {
        "id": 18,
        "title": "EMJM – Selection Criteria",
        "content": (
            "Selection is competitive and based on academic excellence, motivation, relevant experience, "
            "language skills, and quality of recommendation letters. Some programs require interviews or tests."
        )
    },
    {
        "id": 19,
        "title": "EMJM – Required Documents",
        "content": (
            "Common documents: transcripts, CV, passport, English proficiency proof, statement of purpose, "
            "recommendation letters, and sometimes portfolio or project proposals."
        )
    },
    {
        "id": 20,
        "title": "European Solidarity Corps (ESC)",
        "content": (
            "ESC funds volunteer and solidarity projects for young people in EU and partner countries. "
            "Funding covers travel, accommodation, food, insurance, and pocket money."
        )
    },
    {
        "id": 21,
        "title": "CEEPUS Mobility Program",
        "content": (
            "CEEPUS supports academic mobility within Central and Eastern Europe, offering monthly stipends "
            "and in some cases accommodation, depending on the host country."
        )
    },
    {
        "id": 22,
        "title": "Swiss Government Excellence Scholarships",
        "content": (
            "Swiss Excellence Scholarships support international researchers and artists for PhD, postdoc, "
            "or research fellowships in Switzerland, covering stipends and sometimes tuition and insurance."
        )
    },
    {
        "id": 23,
        "title": "ThinkSwiss Research Scholarship",
        "content": (
            "ThinkSwiss offers funding for short research stays in Switzerland. Applicants must show academic "
            "excellence and have an invitation from a Swiss professor or host institution."
        )
    },
    {
        "id": 24,
        "title": "Horizon Europe Research Mobility",
        "content": (
            "Horizon Europe funds large research projects where students and early-stage researchers "
            "may participate through research assistantships or mobility actions like Marie Skłodowska-Curie."
        )
    },
    {
        "id": 25,
        "title": "ERC Projects – Student Participation",
        "content": (
            "ERC grants are awarded to principal investigators, but students can join these teams as research "
            "assistants, interns, or PhD candidates, depending on the host institution."
        )
    },

    # --- USA / UK / Canada / Australia ---
    {
        "id": 26,
        "title": "Fulbright Foreign Student Program – Overview",
        "content": (
            "Fulbright provides fully funded master’s and PhD opportunities in the United States for international "
            "students. It aims to increase mutual understanding between the U.S. and other countries."
        )
    },
    {
        "id": 27,
        "title": "Fulbright – Financial Coverage",
        "content": (
            "Fulbright typically covers tuition, living stipend, health insurance, and round-trip international "
            "travel. Some awards also support books, research, and conference attendance."
        )
    },
    {
        "id": 28,
        "title": "Fulbright – Selection Criteria",
        "content": (
            "Selection emphasizes academic strength, leadership, clear goals, cultural ambassadorship, "
            "and the potential impact of the proposed study or research project."
        )
    },
    {
        "id": 29,
        "title": "Fulbright – Application Documents",
        "content": (
            "Applicants usually submit transcripts, CV, personal statement, study/research objectives, "
            "recommendation letters, English test scores (TOEFL/IELTS), and possibly writing samples."
        )
    },
    {
        "id": 30,
        "title": "Gilman Scholarship – Overview",
        "content": (
            "The Gilman Scholarship supports U.S. undergraduates with financial need to study or intern abroad. "
            "It prioritizes diverse applicants and non-traditional destinations."
        )
    },
    {
        "id": 31,
        "title": "Boren Awards – Overview",
        "content": (
            "Boren Awards fund U.S. students to study critical languages and regions. Recipients have a service "
            "requirement to work in the U.S. government after their studies."
        )
    },
    {
        "id": 32,
        "title": "NSF Graduate Research Fellowship Program (NSF-GRFP)",
        "content": (
            "NSF-GRFP is a prestigious fellowship for early-stage graduate students in STEM in the U.S. "
            "It offers a multi-year stipend and cost-of-education allowance."
        )
    },
    {
        "id": 33,
        "title": "NIH Graduate Partnerships and PhD Programs",
        "content": (
            "NIH collaborates with universities to offer joint PhD and research training programs. "
            "Students conduct research in NIH labs and receive competitive stipends."
        )
    },
    {
        "id": 34,
        "title": "Chevening Scholarship – Overview",
        "content": (
            "Chevening is the UK government’s fully funded one-year master’s scholarship for emerging leaders "
            "from around the world."
        )
    },
    {
        "id": 35,
        "title": "Chevening – Eligibility and Experience Requirements",
        "content": (
            "Chevening typically requires an undergraduate degree equivalent to a UK 2:1 and at least two years "
            "of work experience, including paid work, internships, or volunteering. Recipients must return home "
            "for at least two years after the award."
        )
    },
    {
        "id": 36,
        "title": "Chevening – Application Components",
        "content": (
            "Applications include four essays, two references, transcripts, and up to three UK master’s program "
            "choices. Shortlisted candidates attend an interview at the British embassy or high commission."
        )
    },
    {
        "id": 37,
        "title": "Commonwealth Scholarship – Overview",
        "content": (
            "Commonwealth Scholarships fund master’s and PhD study in the UK for students from eligible "
            "Commonwealth countries, focusing on development impact and academic excellence."
        )
    },
    {
        "id": 38,
        "title": "Rhodes Scholarship – Overview",
        "content": (
            "The Rhodes Scholarship funds postgraduate study at the University of Oxford for outstanding students "
            "who demonstrate academic excellence, leadership, and commitment to service."
        )
    },
    {
        "id": 39,
        "title": "Gates Cambridge Scholarship – Overview",
        "content": (
            "Gates Cambridge supports excellent non-UK students to pursue full-time postgraduate degrees at the "
            "University of Cambridge, covering full tuition and a living allowance."
        )
    },
    {
        "id": 40,
        "title": "Vanier Canada Graduate Scholarships – Overview",
        "content": (
            "Vanier CGS offers generous multi-year funding to attract world-class doctoral students to Canada "
            "across health, science, engineering, social sciences, and humanities."
        )
    },
    {
        "id": 41,
        "title": "Banting Postdoctoral Fellowships – Overview",
        "content": (
            "Banting Fellowships fund top postdoctoral researchers in Canada or abroad, with an emphasis on "
            "leadership and research excellence."
        )
    },
    {
        "id": 42,
        "title": "Mitacs Globalink Research Internship – Overview",
        "content": (
            "Mitacs Globalink offers a fixed 12-week research internship in Canada for undergraduates from partner "
            "countries (minimum 10 weeks in some cases), including stipends and travel support."
        )
    },
    {
        "id": 43,
        "title": "Australia Awards Scholarships – Overview",
        "content": (
            "Australia Awards fund students from selected countries for full-time undergrad or postgrad study "
            "in Australia, focusing on development priorities."
        )
    },
    {
        "id": 44,
        "title": "Australia Awards – Coverage and Obligations",
        "content": (
            "The award usually covers tuition, flights, establishment allowance, living expenses, and health "
            "insurance. Graduates are expected to apply their skills to development in their home country."
        )
    },
    {
        "id": 45,
        "title": "Destination Australia Program – Overview",
        "content": (
            "The Destination Australia Program provides scholarships to encourage study in regional Australia "
            "for both domestic and international students."
        )
    },

    # --- Asia + Middle East + Africa + Global ---
    {
        "id": 46,
        "title": "MEXT Scholarship – Overview",
        "content": (
            "MEXT is the Japanese government scholarship for undergraduate, master’s, PhD, and research students. "
            "It commonly covers tuition, monthly stipend, and round-trip airfare."
        )
    },
    {
        "id": 47,
        "title": "MEXT – Eligibility and Selection",
        "content": (
            "MEXT applicants must satisfy academic, age, and health criteria. Selection includes embassy screening, "
            "exams, and interviews, and research plans for graduate-level applicants."
        )
    },
    {
        "id": 48,
        "title": "Global Korea Scholarship (GKS) – Overview",
        "content": (
            "GKS is a fully funded South Korean government scholarship supporting undergraduate and graduate "
            "degrees, including a year of Korean language training in most cases."
        )
    },
    {
        "id": 49,
        "title": "GKS – Academic and Language Requirements",
        "content": (
            "GKS requires strong academic records and GPA above thresholds defined by each call. TOPIK scores "
            "improve competitiveness, and some programs require a certain Korean proficiency level."
        )
    },
    {
        "id": 50,
        "title": "Chinese Government Scholarship (CSC) – Overview",
        "content": (
            "CSC funds international students for bachelor’s, master’s, and PhD programs in China, covering "
            "tuition, housing, medical insurance, and a monthly stipend."
        )
    },
    {
        "id": 51,
        "title": "Taiwan ICDF Scholarship Program – Overview",
        "content": (
            "The Taiwan ICDF scholarship supports fully funded master’s and PhD studies in development-related "
            "fields, covering tuition, airfare, housing, books, insurance, and a stipend."
        )
    },
    {
        "id": 52,
        "title": "Singapore A*STAR Graduate Scholarship – Overview",
        "content": (
            "A*STAR offers PhD scholarships in science and engineering, including tuition, monthly stipend, and "
            "access to advanced research facilities in Singapore."
        )
    },
    {
        "id": 53,
        "title": "Hong Kong PhD Fellowship Scheme – Overview",
        "content": (
            "The Hong Kong PhD Fellowship funds top international students for doctoral studies with a generous "
            "annual stipend and conference travel allowance."
        )
    },
    {
        "id": 54,
        "title": "Qatar University Graduate Scholarships – Overview",
        "content": (
            "Qatar University offers internal and external graduate scholarships for masters and PhD programs, "
            "covering tuition and, in some cases, monthly stipends."
        )
    },
    {
        "id": 55,
        "title": "UAE University Graduate Assistantships – Overview",
        "content": (
            "UAEU grants graduate assistantships to master's and PhD students, providing tuition waivers and "
            "monthly stipends in exchange for teaching or research duties."
        )
    },
    {
        "id": 56,
        "title": "KAUST Fellowship (Saudi Arabia) – Overview",
        "content": (
            "KAUST offers fully funded fellowships for graduate study in science and engineering, including "
            "tuition, monthly living allowance, housing, and relocation support."
        )
    },
    {
        "id": 57,
        "title": "KFUPM Graduate Scholarship – Overview",
        "content": (
            "King Fahd University of Petroleum and Minerals provides fully funded scholarships for master's and "
            "PhD degrees, covering tuition, stipend, housing, and medical care."
        )
    },
    {
        "id": 58,
        "title": "King Abdullah Scholarship Program (KASP, Saudi Arabia) – Overview",
        "content": (
            "KASP supports Saudi citizens to study abroad in undergraduate, master’s, PhD, and medical programs. "
            "It funds tuition, living costs, and travel in priority fields."
        )
    },
    {
        "id": 59,
        "title": "Türkiye Scholarships (YTB) – Overview",
        "content": (
            "Türkiye Scholarships fund international students for undergraduate, master’s, and PhD study in Türkiye, "
            "including tuition, housing, health insurance, monthly stipends, and Turkish language training."
        )
    },
    {
        "id": 60,
        "title": "Stipendium Hungaricum – Overview",
        "content": (
            "Stipendium Hungaricum is a Hungarian government scholarship offering full or partial funding for "
            "bachelor’s, master’s, doctoral, and preparatory programs."
        )
    },
    {
        "id": 61,
        "title": "Italian Regional Scholarships – Overview",
        "content": (
            "Several Italian regions and universities provide need-based scholarships with tuition waivers, housing "
            "support, and meal subsidies for high-achieving students."
        )
    },
    {
        "id": 62,
        "title": "New Zealand Manaaki Scholarships – Overview",
        "content": (
            "Manaaki Scholarships support students from developing countries to undertake full-time study in "
            "New Zealand, with tuition, living stipend, and travel costs covered."
        )
    },
    {
        "id": 63,
        "title": "African Union Scholarship Programs – Overview",
        "content": (
            "The African Union supports master’s and PhD students in priority fields to build scientific and "
            "technical capacity across the continent."
        )
    },
    {
        "id": 64,
        "title": "Mastercard Foundation Scholars Program – Overview",
        "content": (
            "The Mastercard Foundation Scholars Program supports talented but economically disadvantaged students "
            "from Africa at partner universities, focusing on leadership and community impact."
        )
    },
    {
        "id": 65,
        "title": "Mandela Rhodes Scholarship – Overview",
        "content": (
            "The Mandela Rhodes Scholarship funds postgraduate study for young African leaders, combining "
            "academic funding with leadership development."
        )
    },
    {
        "id": 66,
        "title": "ASEAN Scholarship – Overview",
        "content": (
            "The ASEAN Scholarship supports students from Southeast Asian countries to study in Singapore, "
            "covering tuition, accommodation, and living allowance."
        )
    },
    {
        "id": 67,
        "title": "Islamic Development Bank (IsDB) Scholarship – Overview",
        "content": (
            "IsDB scholarships fund bachelor’s, master’s, PhD, and postdoctoral studies for students from member "
            "countries in development-related fields."
        )
    },
    {
        "id": 68,
        "title": "Russia Open Doors Scholarship – Overview",
        "content": (
            "Open Doors Olympiad offers international students the chance to win fully funded master’s and PhD "
            "places at Russian universities based on academic competition performance."
        )
    },
    {
        "id": 69,
        "title": "Latin America Mobility Programs – Overview",
        "content": (
            "Countries such as Brazil, Mexico, and Argentina run government or bilateral mobility programs for "
            "student exchange. Funding levels vary by agreement."
        )
    },
    {
        "id": 70,
        "title": "Oman and Kuwait Government Scholarships – Overview",
        "content": (
            "Oman and Kuwait provide government-funded scholarships for citizens to study abroad, typically "
            "covering tuition, living expenses, and travel in designated fields and institutions."
        )
    },
]

print(f"Knowledge base loaded with {len(knowledge_base)} documents.")



Knowledge base loaded with 70 documents.


In [3]:
# ============================================================
# Embeddings & FAISS Index
# ============================================================

# Prepare texts
document_texts: List[str] = [
    f"{doc['title']}. {doc['content']}" for doc in knowledge_base
]

# Embedding model
embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedder = SentenceTransformer(embedding_model_name)

# Compute embeddings
document_embeddings = embedder.encode(document_texts, convert_to_numpy=True)
embedding_dim = document_embeddings.shape[1]

# Build FAISS index
index = faiss.IndexFlatL2(embedding_dim)
index.add(document_embeddings)

print("FAISS index built.")
print("Number of vectors in index:", index.ntotal)


FAISS index built.
Number of vectors in index: 70


In [4]:
# ============================================================
# LLM Setup – FLAN-T5 Small (seq2seq model)
# ============================================================

model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
llm_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

text_generator = pipeline(
    "text2text-generation",   # important for T5-style models
    model=llm_model,
    tokenizer=tokenizer,
    max_new_tokens=200
)

print("FLAN-T5-Small pipeline initialized.")


Device set to use cpu


FLAN-T5-Small pipeline initialized.


In [5]:
# ============================================================
# Baseline: answer WITHOUT RAG (no retrieval, model only)
# ============================================================

def generate_answer_no_rag(query: str) -> str:
    """
    Baseline: FLAN-T5 answers directly, without using our knowledge base.
    This simulates the 'no RAG' scenario.
    """
    prompt = f"""
You are an assistant that answers questions about scholarships and student mobility programs.
Answer the following question as best as you can using your own knowledge.

Question: {query}
Answer:
""".strip()

    output = text_generator(prompt)[0]["generated_text"]
    return output.strip()


In [6]:
# ============================================================
# Prompt Builder
# ============================================================

def build_prompt(context: str, query: str) -> str:
    """
    Build a prompt specialized for scholarships & mobility programs.
    The model must answer ONLY using the given context.
    """
    prompt = f"""
You are an assistant that answers questions about scholarship and mobility programs
around the world. These include Erasmus, Mevlana, Fulbright, DAAD,
Chevening, MEXT, and many others.

RULES:
- Use ONLY the information from the context below.
- If the answer is not clearly in the context, say:
  "I am not sure based on this data. Please check the official website for up-to-date details."
- Do NOT invent exact GPA thresholds, deadlines, or grant amounts not mentioned.
- Answer in a concise, student-friendly way.

Context:
{context}

Question:
{query}

Answer:
"""
    return prompt.strip()


In [7]:
# ============================================================
# Retrieval and Answer Generation
# ============================================================

def retrieve_documents(query: str, top_k: int = 5) -> List[Dict]:
    """
    Retrieve top-k documents using FAISS based on semantic similarity.
    """
    query_embedding = embedder.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_embedding, top_k)

    results: List[Dict] = []
    for rank, idx in enumerate(indices[0]):
        doc = knowledge_base[int(idx)]
        results.append({
            "rank": rank + 1,
            "id": doc["id"],
            "title": doc["title"],
            "content": doc["content"],
            "distance": float(distances[0][rank])
        })
    return results


def generate_answer(context: str, query: str) -> str:
    """
    Generate an answer using the LLM and the retrieved context.
    """
    prompt = build_prompt(context, query)
    output = text_generator(prompt)[0]["generated_text"]

    # Try to return only the answer part after "Answer:"
    if "Answer:" in output:
        answer = output.split("Answer:", 1)[-1].strip()
    else:
        answer = output.strip()

    return answer


In [8]:
# ============================================================
# Full RAG Pipeline
# ============================================================

def rag_pipeline(query: str, top_k: int = 5) -> Dict:
    """
    1. Retrieve relevant docs
    2. Build context
    3. Generate answer
    """
    retrieved = retrieve_documents(query, top_k=top_k)
    context_parts = [f"[{d['title']}]\n{d['content']}" for d in retrieved]
    context = "\n\n".join(context_parts)

    answer = generate_answer(context, query)

    return {
        "query": query,
        "retrieved": retrieved,
        "context": context,
        "answer": answer
    }


In [9]:
# ============================================================
# Comparison: top_k = 3 vs top_k = 5
# ============================================================

comparison_questions = [
    "What is Erasmus Mundus Joint Master’s?",
    "What does Fullbright usually cover for students?",
    "What is the Global Korea Scholarship (GKS)?",
    "What is Türkiye Scholarships (YTB)?"
]

for q in comparison_questions:
    print("=" * 120)
    print("Question:", q, "\n")

    # 1) Baseline (no RAG)
    no_rag_ans = generate_answer_no_rag(q)
    print("WITHOUT RAG (model only):")
    print(textwrap.fill(no_rag_ans, width=100))
    print()

    # 2) RAG with k = 3
    rag_result_k3 = rag_pipeline(q, top_k=3)
    rag_ans_k3 = rag_result_k3["answer"]
    print("WITH RAG (top_k = 3):")
    print(textwrap.fill(rag_ans_k3, width=100))
    print()

    # 3) RAG with k = 5
    rag_result_k5 = rag_pipeline(q, top_k=5)
    rag_ans_k5 = rag_result_k5["answer"]
    print("WITH RAG (top_k = 5):")
    print(textwrap.fill(rag_ans_k5, width=100))
    print()

    print("Retrieved docs (k = 3):")
    for d in rag_result_k3["retrieved"]:
        print(f"  - {d['title']} (dist={d['distance']:.4f})")
    print()

    print("Retrieved docs (k = 5):")
    for d in rag_result_k5["retrieved"]:
        print(f"  - {d['title']} (dist={d['distance']:.4f})")
    print("\n")


Question: What is Erasmus Mundus Joint Master’s? 

WITHOUT RAG (model only):
Erasmus Mundus Joint Master

WITH RAG (top_k = 3):
EMJM programs are joint master's degrees offered by consortia of European universities. Students
study in at least two countries and receive a joint or multiple degree. [Erasmus – Program Overview]
Erasmus is a European student mobility program enabling students to study or intern abroad for one
or two semesters. It promotes cultural exchange, academic development, and cooperation. [EMJM –
Financial Package] Erasmus Mundus scholarships usually cover full tuition fees, a monthly stipend,
travel costs, and installation support. The exact amounts and rules vary by consortium.

WITH RAG (top_k = 5):
EMJM programs are joint master's degrees offered by consortia of European universities. Students
study in at least two countries and receive a joint or multiple degree. [Erasmus – Program Overview]
Erasmus is a European student mobility program enabling students to stu